# 实验三参考答案：Python 模块化与科学数据处理

教师用文件。完整函数实现位于 `data_utils_参考答案.py`。

In [ ]:
from pathlib import Path
import platform
import sys

import numpy as np
import pandas as pd

import data_utils_参考答案 as data_utils

print(sys.version)
print(sys.executable)
print(platform.platform())
print(np.__version__, pd.__version__)

## 模块来源练习

In [ ]:
module_sources = {
    "__future__": "future",
    "pathlib": "standard-library",
    "numpy": "third-party",
    "data_utils": "project-code",
}
assert set(module_sources.values()) == {
    "future", "standard-library", "third-party", "project-code"
}

## NumPy 练习

In [ ]:
features = np.array(
    [
        [5.1, 3.5, 1.4],
        [4.9, 3.0, 1.4],
        [7.0, 3.2, 4.7],
        [6.4, 3.2, 4.5],
    ]
)
feature_shape = features.shape
feature_ndim = features.ndim
feature_size = features.size
second_column = features[:, 1]
selected_rows = features[features[:, 0] >= 6.0]
means_by_feature = features.mean(axis=0)
sums_by_sample = features.sum(axis=1)
feature_means = features.mean(axis=0)
feature_stds = features.std(axis=0)
standardized_features = (features - feature_means) / feature_stds

assert feature_shape == (4, 3)
assert feature_ndim == 2 and feature_size == 12
assert np.array_equal(second_column, [3.5, 3.0, 3.2, 3.2])
assert selected_rows.shape == (2, 3)
assert np.allclose(means_by_feature, [5.85, 3.225, 3.0])
assert np.allclose(sums_by_sample, [10.0, 9.3, 14.9, 14.1])
assert np.allclose(standardized_features.mean(axis=0), 0.0)
assert np.allclose(standardized_features.std(axis=0), 1.0)
print("✅ NumPy 练习通过")

## Pandas 练习

In [ ]:
data_path = Path("data") / "samples.csv"
raw_data = pd.read_csv(data_path)
row_count, column_count = raw_data.shape
missing_cell_count = int(raw_data.isna().sum().sum())
duplicate_row_count = int(raw_data.duplicated().sum())
selected_data = raw_data.loc[
    (raw_data["label"] == "B") & (raw_data["feature_1"] > 6.0),
    ["sample_id", "feature_1", "feature_2", "feature_3"],
]

assert (row_count, column_count) == (16, 5)
assert missing_cell_count == 2 and duplicate_row_count == 1
assert selected_data["sample_id"].tolist() == ["S004", "S005", "S006", "S014"]
display(selected_data)

## 综合实验

In [ ]:
raw = data_utils.load_data(data_path)
raw_snapshot = raw.copy(deep=True)
quality_report = data_utils.inspect_data(raw)
cleaned = data_utils.clean_data(raw)
standardized = data_utils.standardize_features(cleaned)
summary = data_utils.summarize_by_label(cleaned)
unusual_sample_id = data_utils.find_most_unusual_sample(cleaned)

display(cleaned)
display(standardized)
display(summary)
print(quality_report)
print("最不寻常样本：", unusual_sample_id)

In [ ]:
assert quality_report == {
    "rows": 16, "columns": 5, "missing_cells": 2, "duplicate_rows": 1
}
assert cleaned.shape == (11, 5)
assert cleaned["label"].value_counts().to_dict() == {"B": 5, "A": 3, "C": 3}
z_columns = [f"z_{column}" for column in data_utils.FEATURE_COLUMNS]
assert np.allclose(standardized[z_columns].mean().to_numpy(), 0.0)
assert np.allclose(standardized[z_columns].std(ddof=0).to_numpy(), 1.0)
assert summary["sample_count"].tolist() == [3, 5, 3]
assert unusual_sample_id == "S001"
pd.testing.assert_frame_equal(raw, raw_snapshot)

try:
    data_utils.clean_data(raw.drop(columns=["label"]))
except ValueError:
    pass
else:
    raise AssertionError("缺少必需列时应抛出 ValueError")

constant_data = cleaned.copy()
constant_data["feature_1"] = 1.0
try:
    data_utils.standardize_features(constant_data)
except ValueError:
    pass
else:
    raise AssertionError("标准差为 0 时应抛出 ValueError")

print("✅ 参考答案全部自测通过")

## 简答题参考要点

1. `axis=0` 沿样本轴聚合，该轴消失，留下每个特征一个结果。
2. 复制可以保留原始输入，避免函数产生调用者不可见的副作用。
3. 教学上的项目代码合并了 Ruff 的 first-party 与 local-folder；本实验主要使用前者。
4. 浮点运算存在舍入误差，近似相等比逐位完全相等更合适。